# Proyecto Final - Conditional Imitation Learning - Equipo 19

Entrenamiento reproducible para los comandos **STRAIGHT (0)**,
**LEFT (1)** y **RIGHT (2)**. La division entrenamiento/validacion se
realiza antes del aumento para impedir fuga entre una imagen y su reflejo.

In [1]:
# Primera celda de codigo requerida por la actividad: clonar el dataset desde GitHub.
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/xak47d/proyecto-final-cil-equipo19.git"
if IN_COLAB:
    PROJECT_ROOT = Path("/content/proyecto-final-cil-equipo19")
    if not PROJECT_ROOT.exists():
        !git clone https://github.com/xak47d/proyecto-final-cil-equipo19.git /content/proyecto-final-cil-equipo19
    !pip install -q -r /content/proyecto-final-cil-equipo19/requirements_colab.txt
else:
    PROJECT_ROOT = Path.cwd()

print("Proyecto:", PROJECT_ROOT)

Proyecto: /content/proyecto-final-cil-equipo19


## 1. Configuracion reproducible

Se fijan semillas y se centralizan dimensiones, lote y rutas. El
preprocesamiento se replica exactamente en el controlador de Webots.

In [2]:
import json, math, random, shutil
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.layers import Input, Conv2D, Dense, Flatten, Dropout, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Sequence, to_categorical

SEED = 19
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_H, IMG_W = 80, 160
BATCH_SIZE = 32
EPOCHS = 12
COMMANDS = {0: "STRAIGHT", 1: "LEFT", 2: "RIGHT"}

RAW_DIR = PROJECT_ROOT / "dataset"
RAW_IMAGES = RAW_DIR / "images"
RAW_CSV = RAW_DIR / "driving_log.csv"
AUG_DIR = PROJECT_ROOT / "dataset_augmented"
AUG_IMAGES = AUG_DIR / "images"
AUG_CSV = AUG_DIR / "driving_log_augmented.csv"
FIGURES = PROJECT_ROOT / "docs" / "figures"
MODEL_DIR = PROJECT_ROOT / "controllers" / "cil_autonomous_driver"
FIGURES.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## 2. Auditoria del dataset original

Se valida el esquema, los tres comandos permitidos y la correspondencia
uno-a-uno entre registros e imagenes antes de entrenar.

In [3]:
raw_df = pd.read_csv(RAW_CSV)
assert list(raw_df.columns) == ["image", "steering_angle", "command"]
assert set(raw_df["command"].unique()) == set(COMMANDS)
assert raw_df["image"].is_unique
missing = [name for name in raw_df["image"] if not (RAW_IMAGES / name).exists()]
assert not missing, f"Imagenes faltantes: {missing[:5]}"

raw_df["source_id"] = raw_df["image"].map(lambda name: Path(name).stem)
print("Registros originales:", len(raw_df))
print(raw_df["command"].map(COMMANDS).value_counts())
display(raw_df.head())

counts = raw_df["command"].map(COMMANDS).value_counts().reindex(COMMANDS.values())
ax = counts.plot(kind="bar", color=["#2E74B5", "#70AD47", "#ED7D31"])
ax.set_title("Distribucion original por comando")
ax.set_ylabel("Imagenes")
ax.set_xlabel("")
plt.tight_layout()
plt.savefig(FIGURES / "command_distribution_original.png", dpi=180)
plt.show()

Registros originales: 6129
command
RIGHT       3032
LEFT        1895
STRAIGHT    1202
Name: count, dtype: int64


,image,steering_angle,command,source_id
0,img_1782317332.png,0.0,0,img_1782317332
1,img_1782317333.png,0.0,0,img_1782317333
2,img_1782317334.png,0.0,0,img_1782317334
3,img_1782317335.png,0.0,0,img_1782317335
4,img_1782317336.png,0.0,0,img_1782317336


/var/folders/50/pzhcqw5j5212_pw_y99p2ky80000gn/T/ipykernel_87500/1690021843.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Division por fuente y aumento

La division estratificada ocurre sobre las imagenes originales. Solo
despues se crean reflejos; LEFT y RIGHT se intercambian al reflejar.
Para equilibrar STRAIGHT se agregan dos variantes de brillo.

In [4]:
train_raw, val_raw = train_test_split(
    raw_df,
    test_size=0.20,
    random_state=SEED,
    stratify=raw_df["command"],
)
train_sources = set(train_raw["source_id"])
val_sources = set(val_raw["source_id"])
assert train_sources.isdisjoint(val_sources)

expected_signature = {
    "seed": SEED,
    "raw_records": len(raw_df),
    "train_sources": len(train_raw),
    "validation_sources": len(val_raw),
    "commands": COMMANDS,
}
signature_path = AUG_DIR / "augmentation_signature.json"
rebuild = True
if signature_path.exists() and AUG_CSV.exists():
    rebuild = json.loads(signature_path.read_text()) != expected_signature

if rebuild:
    if AUG_DIR.exists():
        shutil.rmtree(AUG_DIR)
    AUG_IMAGES.mkdir(parents=True, exist_ok=True)
    augmented_rows = []

    def save_variant(row, split, variant, image, steering, command):
        output_name = f"{split}_{variant}_{row.image}"
        ok = cv2.imwrite(str(AUG_IMAGES / output_name), image)
        if not ok:
            raise IOError(f"No se pudo escribir {output_name}")
        augmented_rows.append({
            "image": output_name,
            "steering_angle": float(steering),
            "command": int(command),
            "source_id": row.source_id,
            "split": split,
            "variant": variant,
        })

    for row in train_raw.itertuples(index=False):
        image = cv2.imread(str(RAW_IMAGES / row.image))
        save_variant(row, "train", "original", image, row.steering_angle, row.command)
        flipped_command = 2 if row.command == 1 else 1 if row.command == 2 else 0
        save_variant(row, "train", "flip", cv2.flip(image, 1), -row.steering_angle, flipped_command)
        if row.command == 0:
            darker = cv2.convertScaleAbs(image, alpha=0.75, beta=0)
            brighter_flip = cv2.convertScaleAbs(cv2.flip(image, 1), alpha=1.20, beta=0)
            save_variant(row, "train", "dark", darker, row.steering_angle, 0)
            save_variant(row, "train", "bright_flip", brighter_flip, -row.steering_angle, 0)

    for row in val_raw.itertuples(index=False):
        image = cv2.imread(str(RAW_IMAGES / row.image))
        save_variant(row, "validation", "original", image, row.steering_angle, row.command)

    augmented_df = pd.DataFrame(augmented_rows)
    AUG_DIR.mkdir(parents=True, exist_ok=True)
    augmented_df.to_csv(AUG_CSV, index=False)
    signature_path.write_text(json.dumps(expected_signature, indent=2))
else:
    augmented_df = pd.read_csv(AUG_CSV)

train_df = augmented_df.query("split == 'train'").reset_index(drop=True)
val_df = augmented_df.query("split == 'validation'").reset_index(drop=True)
assert set(train_df.source_id).isdisjoint(set(val_df.source_id))
assert len(augmented_df) > 10_000
print("Muestras finales:", len(augmented_df))
print("Entrenamiento:", len(train_df), "Validacion:", len(val_df))
print(train_df["command"].map(COMMANDS).value_counts())

Muestras finales: 12956
Entrenamiento: 11730 Validacion: 1226
command
LEFT        3941
RIGHT       3941
STRAIGHT    3848
Name: count, dtype: int64


## 4. Generador y preprocesamiento

La camara se recorta 25% desde arriba para reducir cielo/edificios,
cambia a RGB, redimensiona a 160x80 y normaliza a [0, 1].

In [5]:
def preprocess_image(path):
    image = cv2.imread(str(path))
    if image is None:
        raise FileNotFoundError(path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image[int(image.shape[0] * 0.25):, :, :]
    image = cv2.resize(image, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    return image.astype(np.float32) / 255.0

class CILSequence(Sequence):
    def __init__(self, dataframe, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.df = dataframe.reset_index(drop=True)
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / BATCH_SIZE))

    def __getitem__(self, index):
        ids = self.indexes[index * BATCH_SIZE:(index + 1) * BATCH_SIZE]
        batch = self.df.iloc[ids]
        images = np.stack([preprocess_image(AUG_IMAGES / name) for name in batch.image])
        commands = to_categorical(batch.command.astype(int), num_classes=3)
        steering = batch.steering_angle.to_numpy(dtype=np.float32)
        return (images, commands), steering

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

train_gen = CILSequence(train_df, shuffle=True)
val_gen = CILSequence(val_df, shuffle=False)
sample_inputs, sample_targets = train_gen[0]
print(sample_inputs[0].shape, sample_inputs[1].shape, sample_targets.shape)

(32, 80, 160, 3) (32, 3) (32,)


## 5. CNN condicionada

La rama visual comparte cinco capas convolucionales. El comando one-hot
se concatena antes de las capas densas para condicionar el angulo.

In [6]:
image_input = Input(shape=(IMG_H, IMG_W, 3), name="image_input")
x = Conv2D(24, 5, strides=2, activation="relu")(image_input)
x = Conv2D(36, 5, strides=2, activation="relu")(x)
x = Conv2D(48, 5, strides=2, activation="relu")(x)
x = Conv2D(64, 3, activation="relu")(x)
x = Conv2D(64, 3, activation="relu")(x)
x = Flatten()(x)

command_input = Input(shape=(3,), name="command_input")
x = Concatenate()([x, command_input])
x = Dense(100, activation="relu")(x)
x = Dropout(0.30)(x)
x = Dense(50, activation="relu")(x)
x = Dense(10, activation="relu")(x)
steering_output = Dense(1, name="steering_output")(x)

model = Model([image_input, command_input], steering_output, name="CIL_Equipo19")
model.compile(optimizer=Adam(1e-4), loss="mse", metrics=["mae"])
model.summary()

Model: "CIL_Equipo19"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 80, 160,   │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 38, 78,    │      1,824 │ image_input[0][0] │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 17, 37,    │     21,636 │ conv2d[0][0]      │
│                     │ 36)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 7, 17, 48) │     43,248 │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 5, 15, 64) │     27,712 │ conv2d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 3, 13, 64) │     36,928 │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 2496)      │          0 │ conv2d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ command_input       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 2499)      │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ command_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 100)       │    250,000 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 100)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 50)        │      5,050 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 10)        │        510 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ steering_output     │ (None, 1)         │         11 │ dense_2[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 386,919 (1.48 MB)

 Trainable params: 386,919 (1.48 MB)

 Non-trainable params: 0 (0.00 B)

## 6. Entrenamiento con callbacks

In [7]:
checkpoint = MODEL_DIR / "cil_model_best.h5"
callbacks = [
    EarlyStopping(monitor="val_mae", patience=3, restore_best_weights=True, mode="min"),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    ModelCheckpoint(checkpoint, monitor="val_mae", save_best_only=True, mode="min"),
]
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/12


  1/367 ━━━━━━━━━━━━━━━━━━━━ 5:43 940ms/step - loss: 0.0069 - mae: 0.0612

  3/367 ━━━━━━━━━━━━━━━━━━━━ 18s 51ms/step - loss: 0.0082 - mae: 0.0607  

  4/367 ━━━━━━━━━━━━━━━━━━━━ 19s 53ms/step - loss: 0.0084 - mae: 0.0625

  5/367 ━━━━━━━━━━━━━━━━━━━━ 20s 57ms/step - loss: 0.0081 - mae: 0.0620

...

364/367 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - loss: 0.0010 - mae: 0.0213

365/367 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - loss: 0.0010 - mae: 0.0213

366/367 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - loss: 0.0010 - mae: 0.0213

367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - loss: 0.0010 - mae: 0.0213

367/367 ━━━━━━━━━━━━━━━━━━━━ 32s 86ms/step - loss: 0.0010 - mae: 0.0213 - val_loss: 0.0012 - val_mae: 0.0220 - learning_rate: 1.0000e-04


## 7. Curvas y evaluacion por comando

In [8]:
history_df = pd.DataFrame(history.history)
history_df.to_csv(FIGURES / "training_history.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history_df.loss, label="train")
axes[0].plot(history_df.val_loss, label="validation")
axes[0].set_title("MSE")
axes[0].set_xlabel("Epoca")
axes[0].legend()
axes[1].plot(history_df.mae, label="train")
axes[1].plot(history_df.val_mae, label="validation")
axes[1].axhline(0.05, color="red", linestyle="--", label="objetivo 0.05")
axes[1].set_title("MAE (rad)")
axes[1].set_xlabel("Epoca")
axes[1].legend()
plt.tight_layout()
plt.savefig(FIGURES / "training_curves.png", dpi=180)
plt.show()

predictions = model.predict(val_gen, verbose=0).reshape(-1)
actual = val_df.steering_angle.to_numpy(dtype=np.float32)
evaluation = []
for command, name in COMMANDS.items():
    mask = val_df.command.to_numpy() == command
    evaluation.append({
        "command": name,
        "samples": int(mask.sum()),
        "mae_rad": float(np.mean(np.abs(predictions[mask] - actual[mask]))),
    })
metrics = {
    "validation_mae_rad": float(np.mean(np.abs(predictions - actual))),
    "validation_mse": float(np.mean(np.square(predictions - actual))),
    "per_command": evaluation,
    "augmented_samples": int(len(augmented_df)),
    "train_samples": int(len(train_df)),
    "validation_samples": int(len(val_df)),
    "source_overlap": 0,
}
print(json.dumps(metrics, indent=2))
display(pd.DataFrame(evaluation))

/var/folders/50/pzhcqw5j5212_pw_y99p2ky80000gn/T/ipykernel_87500/3378205313.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


{
  "validation_mae_rad": 0.022006014361977577,
  "validation_mse": 0.001179361017420888,
  "per_command": [
    {
      "command": "STRAIGHT",
      "samples": 240,
      "mae_rad": 0.01518306229263544
    },
    {
      "command": "LEFT",
      "samples": 379,
      "mae_rad": 0.024078940972685814
    },
    {
      "command": "RIGHT",
      "samples": 607,
      "mae_rad": 0.02340942621231079
    }
  ],
  "augmented_samples": 12956,
  "train_samples": 11730,
  "validation_samples": 1226,
  "source_overlap": 0
}


,command,samples,mae_rad
0,STRAIGHT,240,0.015183
1,LEFT,379,0.024079
2,RIGHT,607,0.023409


## 8. Exportacion para Webots

In [9]:
final_model = MODEL_DIR / "cil_model.h5"
model.save(final_model)
metadata = {
    "image_height": IMG_H,
    "image_width": IMG_W,
    "channels": 3,
    "crop_top_ratio": 0.25,
    "normalization": "uint8/255.0",
    "commands": {str(key): value for key, value in COMMANDS.items()},
    "seed": SEED,
    "validation_mae_target_rad": 0.05,
}
(MODEL_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2))
(MODEL_DIR / "training_metrics.json").write_text(json.dumps(metrics, indent=2))
print("Modelo:", final_model)
print("MAE validacion:", metrics["validation_mae_rad"])
assert metrics["validation_mae_rad"] <= 0.05, "El modelo no alcanza el objetivo MAE"

Modelo: /content/proyecto-final-cil-equipo19/controllers/cil_autonomous_driver/cil_model.h5
MAE validacion: 0.022006014361977577


## Conclusion

El artefacto exportado conserva tres comandos verificables y un
preprocesamiento documentado. La validacion final se realiza en World 2
mediante las rutas recta, derecha e izquierda y el arbitro de seguridad.